# Phase 2.1 - 3-qubit bit-flip repetition code

Our first real QEC code. The encoding is `a|0> + b|1> -> a|000> + b|111>`. We will:

1. Define the code, its stabilizers, and its logical operators.
2. Apply single-qubit depolarizing noise to each physical qubit.
3. Extract a syndrome and apply the ML single-qubit X recovery.
4. Sweep `p` and plot the logical error rate against the uncoded baseline.

**Conceptual limitation flagged up front:** this code corrects bit-flip (X) errors only. Phase (Z) errors commute with both stabilizers and are undetectable. Phase 2.2 (Shor 9) handles both, by concatenating this code with its Hadamard-rotated dual (the phase-flip code).

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from qec_project.codes.repetition3 import ThreeQubitBitFlipCode
from qec_project.noise.quantum import (
    PAULI_I,
    PAULI_X,
    apply_channel,
    depolarizing_channel,
)

rng = np.random.default_rng(seed=42)
code3 = ThreeQubitBitFlipCode()

## 1. Encoding and the code space

The 2-dim logical Hilbert space is spanned by

    |0_L> = |000>,   |1_L> = |111>.

Encoding `a|0> + b|1>` simply maps to `a|000> + b|111>`. Two CNOTs from a data qubit onto two ancillas implement this on hardware; in the density-matrix picture we can write the result directly.

In [ ]:
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
psi_plus_L = code3.encode(psi_plus)
print('|+_L> =', np.round(psi_plus_L, 4))
print('norm:', float(np.real(psi_plus_L.conj() @ psi_plus_L)))

## 2. Stabilizers and logical operators

The two stabilizer generators are

    S0 = Z (x) Z (x) I,   S1 = I (x) Z (x) Z.

The logical operators (Pauli operators that act on the code space) are

    X_L = X (x) X (x) X,   Z_L = Z (x) Z (x) Z.

Both stabilizers fix `|0_L>` and `|1_L>` (eigenvalue +1) because they consist entirely of `Z`s and the code states are computational-basis vectors with all bits equal.

In [ ]:
S0, S1 = code3.stabilizers()
psi_0L = code3.encode(np.array([1.0, 0.0], dtype=complex))
psi_1L = code3.encode(np.array([0.0, 1.0], dtype=complex))
print('S0 |0_L> = |0_L>?', np.allclose(S0 @ psi_0L, psi_0L))
print('S1 |0_L> = |0_L>?', np.allclose(S1 @ psi_0L, psi_0L))
print('S0 |1_L> = |1_L>?', np.allclose(S0 @ psi_1L, psi_1L))
print('S1 |1_L> = |1_L>?', np.allclose(S1 @ psi_1L, psi_1L))

### Why this code is blind to Z errors

A single Z on, say, qubit 1 commutes with both stabilizers (because each stabilizer is a product of Z operators). The syndrome is therefore `(0, 0)`. But `Z_1` acts as `Z_L` on the code space (since on `|000>` and `|111>` any single Z acts as the all-Z product up to an overall sign), so it flips the logical phase. Result: undetectable logical Z error.

The fix is to encode each physical qubit of an outer phase-flip code in this code. That construction is Shor 9 (Phase 2.2, built by another agent in parallel).

## 3. Noise: independent single-qubit depolarizing

Each physical qubit independently undergoes

    rho -> (1 - p) rho + (p/3)(X rho X + Y rho Y + Z rho Z).

At `p = 3/4` the qubit is replaced by the maximally mixed state `I/2` (each Pauli, including I, applied with weight 1/4). This is the standard `DEPOLARIZE1` parameterization in Stim and Nielsen-Chuang Eq. 8.102. The total channel on the 3-qubit state is the tensor product of the three single-qubit channels.

In [ ]:
p_demo = 0.1
rho_0L = np.outer(psi_0L, psi_0L.conj())
rho_noisy = code3.apply_noise(rho_0L, p=p_demo, rng=rng)
print(f'trace after noise (p={p_demo}):', float(np.real(np.trace(rho_noisy))))
print(f'<000|rho|000>: {float(np.real(rho_noisy[0, 0])):.4f}')
print(f'<111|rho|111>: {float(np.real(rho_noisy[7, 7])):.4f}')
print(f'<001|rho|001>: {float(np.real(rho_noisy[1, 1])):.4f}  (single bit-flip on qubit 2)')
print(f'<010|rho|010>: {float(np.real(rho_noisy[2, 2])):.4f}  (single bit-flip on qubit 1)')

## 4. Syndrome extraction

We measure the two stabilizer generators. Their joint eigenspaces partition the 8-dim Hilbert space into four 2-dim blocks, one per syndrome value:

    (0, 0): {|000>, |111>}     code space (no error or weight-3 error)
    (1, 0): {|100>, |011>}     X on qubit 0
    (1, 1): {|010>, |101>}     X on qubit 1
    (0, 1): {|001>, |110>}     X on qubit 2

We can read the syndrome distribution directly from the Z-basis diagonal of `rho`. On real hardware you would build an ancilla-qubit circuit that measures the same observables without collapsing the data qubits' relative phases. Phase 2.2 walks through that circuit in Qiskit.

In [ ]:
syn = code3.syndrome_extract(rho_noisy, rng)
print(f'sampled syndrome: {syn}')

## 5. Recovery

Maximum-likelihood lookup: a syndrome of `(1, 0)` is most likely caused by a single X on qubit 0, etc. We apply the corresponding Pauli X correction. Two- or three-qubit X errors share syndromes with single-qubit X errors on a different qubit and will therefore be mis-corrected. That is the source of the logical error rate we measure below.

In [ ]:
# Showcase: apply a single X on qubit 1, syndrome-extract, decode, check fidelity.
X1 = np.kron(np.kron(PAULI_I, PAULI_X), PAULI_I)
psi_e = X1 @ psi_0L
rho_e = np.outer(psi_e, psi_e.conj())
syn_e = code3.syndrome_extract(rho_e, rng)
rho_corrected = code3.decode(rho_e, syndrome=syn_e)
print(f'syndrome of X_1 error: {syn_e}')
print('recovered |0_L>?', np.allclose(rho_corrected, np.outer(psi_0L, psi_0L.conj())))

## 6. Logical error rate vs physical error rate

Now the headline plot. We sweep `p` and, for each value, run `N = 2000` trials. Each trial picks one of the four logical basis states `{|0>, |1>, |+>, |->}` (so we are averaging the channel infidelity over a 2-design - the standard average-case metric for a code), encodes, noises, syndrome-extracts, decodes, and checks whether the recovered logical state has fidelity > 0.5 with the input.

The uncoded baseline is one qubit undergoing the same depolarizing channel. To keep apples-to-apples, we run the same Monte-Carlo procedure on the uncoded qubit and read off its empirical failure rate.

Expected shape: at small `p` the coded curve sits below the uncoded curve (coding helps) and at large `p` it eventually crosses (coding hurts because the syndrome lookup mis-corrects too often). The crossover is the code's *pseudo-threshold*. For this toy distance-3 code with this decoder, the curves are close throughout - distance is the same as for the classical Hamming(7,4) plot in Phase 0.3, so this picture is purely qualitative.

In [ ]:
def uncoded_logical_error_rate(
    p: float, n_trials: int, rng: np.random.Generator
) -> float:
    basis = [
        np.array([1.0, 0.0], dtype=complex),
        np.array([0.0, 1.0], dtype=complex),
        np.array([1.0, 1.0], dtype=complex) / np.sqrt(2),
        np.array([1.0, -1.0], dtype=complex) / np.sqrt(2),
    ]
    K = depolarizing_channel(p)
    failures = 0
    for _ in range(n_trials):
        psi = basis[int(rng.integers(0, 4))]
        rho = np.outer(psi, psi.conj())
        rho_n = apply_channel(rho, K)
        fidelity = float(np.real(psi.conj() @ rho_n @ psi))
        if fidelity < 0.5:
            failures += 1
    return failures / n_trials

In [ ]:
ps = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
N = 2000

rng_coded = np.random.default_rng(seed=42)
rng_unc = np.random.default_rng(seed=43)

coded = [code3.simulate_logical_error_rate(p=p, n_trials=N, rng=rng_coded) for p in ps]
uncoded = [uncoded_logical_error_rate(p=p, n_trials=N, rng=rng_unc) for p in ps]

for p, c, u in zip(ps, coded, uncoded, strict=True):
    print(f'p={p:.3f}  coded={c:.4f}  uncoded={u:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(ps, coded, 'o-', label='3-qubit code (with decoder)')
ax.plot(ps, uncoded, 's--', label='uncoded (single qubit)')
ax.plot(ps, ps, ':', color='gray', label='p (reference)')
ax.set_xlabel('physical depolarizing rate p')
ax.set_ylabel('logical error rate (empirical, N=2000)')
ax.set_title('3-qubit bit-flip repetition code under depolarizing noise')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('logical_vs_physical.png', dpi=150)
plt.show()

## 7. Connect the dots

The shape of this plot is the shape of every QEC benchmark you will produce in this project:

- **Phase 0.3** showed the same plot for the classical Hamming(7,4) code under a binary symmetric channel. Same axes, same crossover, same intuition.
- **Phase 3** will show the same plot for the rotated surface code at distances 3, 5, 7 - except those curves will *cross at the threshold*, the same crossing point regardless of distance. That is the headline phenomenon the capstone exists to characterize.

What this code is missing - phase-error correction - motivates Phase 2.2 (Shor 9). What this code is missing - fault-tolerant syndrome extraction - motivates Phase 2.5 (Stim) and Phase 4.